# M4 — XGBoost + Target Encoding + Optuna (version corrigée)

**Stratégie d'encodage :** ordinal pour les qualités ordonnées, **Target Encoding pour toutes les variables nominales** (pas de one-hot : inutilement coûteux en dimensions pour un modèle d'arbres).

**Corrections / améliorations par rapport à la version initiale :**

1. **Bug corrigé** — les colonnes de qualité (`BsmtQual`, `BsmtCond`, `FireplaceQu`, `GarageQual`, `GarageCond`) ne sont plus traitées deux fois.
2. **`KNNImputer` supprimé** — inutile et de mauvaise qualité sans scaling pour un modèle d'arbres. Remplacé par une imputation médiane (numérique), réajustée par pli → pas de fuite.
3. **Manquants structurels gérés selon leur sens métier** — surfaces absentes → `0`, qualités absentes → `0`, catégories absentes → `'None'`.
4. **Nouvelles variables** — `TotalSF`, `TotalBath`, `TotalPorchSF`, `QualxArea`, indicateurs `Has*`, âges bornés à 0.
5. **Early stopping** dans la validation croisée + refit final sur tout le jeu d'entraînement.
6. **Nettoyage** — typo `GarageYrBlt=2207` corrigée, âges négatifs bornés, imports morts retirés.

> ⚠️ Dépendances : `pip install xgboost optuna category_encoders scikit-learn`

In [13]:
# 1. IMPORTATIONS
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

from category_encoders import TargetEncoder
from xgboost import XGBRegressor
import optuna

In [14]:
# 2. FEATURE ENGINEERING
# Transformations DÉTERMINISTES (sans la cible) -> applicables identiquement au train et au test.

def feature_engineering(data):
    df = data.copy()

    # MSSubClass est un CODE de type de logement, pas une quantité -> catégoriel
    df['MSSubClass'] = df['MSSubClass'].astype(str)

    # A. Surfaces/compteurs ABSENTS = 0 (NaN => pas de sous-sol / garage / etc.)
    zero_fill = ['MasVnrArea', 'GarageArea', 'GarageCars', 'TotalBsmtSF',
                 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'BsmtFullBath', 'BsmtHalfBath']
    for c in zero_fill:
        if c in df.columns:
            df[c] = df[c].fillna(0)

    # B. Garage : pas d'année => année de construction ; correction du typo 2207
    df['GarageYrBlt'] = df['GarageYrBlt'].fillna(df['YearBuilt'])
    df['GarageYrBlt'] = df['GarageYrBlt'].clip(upper=df['YrSold'])

    # C. ENCODAGE ORDINAL des qualités (Po < Fa < TA < Gd < Ex)
    qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
    # Equipement pouvant être ABSENT : NaN -> 0
    for c in ['BsmtQual', 'BsmtCond', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']:
        if c in df.columns:
            df[c] = df[c].map(qual_map).fillna(0)
    # Toujours présent : un NaN est un vrai manquant -> valeur typique (TA=3)
    for c in ['ExterQual', 'ExterCond', 'HeatingQC', 'KitchenQual']:
        if c in df.columns:
            df[c] = df[c].map(qual_map).fillna(3)

    # Autres variables ordonnées
    df['BsmtExposure'] = df['BsmtExposure'].map({'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0}).fillna(0)
    fin_map = {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0}
    df['BsmtFinType1'] = df['BsmtFinType1'].map(fin_map).fillna(0)
    df['BsmtFinType2'] = df['BsmtFinType2'].map(fin_map).fillna(0)
    df['GarageFinish'] = df['GarageFinish'].map({'Fin': 3, 'RFn': 2, 'Unf': 1, 'None': 0}).fillna(0)
    df['Functional'] = df['Functional'].map(
        {'Typ': 7, 'Min1': 6, 'Min2': 5, 'Mod': 4, 'Maj1': 3, 'Maj2': 2, 'Sev': 1, 'Sal': 0}).fillna(7)

    # D. NOUVELLES VARIABLES (agrégats + interaction)
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalBath'] = df['FullBath'] + 0.5 * df['HalfBath'] + df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath']
    df['TotalPorchSF'] = (df['OpenPorchSF'] + df['EnclosedPorch'] + df['3SsnPorch']
                          + df['ScreenPorch'] + df['WoodDeckSF'])
    df['QualxArea'] = df['OverallQual'] * df['GrLivArea']
    df['IsRemodeled'] = (df['YearBuilt'] != df['YearRemodAdd']).astype(int)
    df['HasGarage'] = (df['GarageArea'] > 0).astype(int)
    df['HasBsmt'] = (df['TotalBsmtSF'] > 0).astype(int)
    df['HasFireplace'] = (df['Fireplaces'] > 0).astype(int)
    df['HasPool'] = (df['PoolArea'] > 0).astype(int)

    # E. ÂGES (bornés à 0)
    df['AgeBuilt'] = (df['YrSold'] - df['YearBuilt']).clip(lower=0)
    df['AgeRemodAdd'] = (df['YrSold'] - df['YearRemodAdd']).clip(lower=0)
    df['AgeGarage'] = (df['YrSold'] - df['GarageYrBlt']).clip(lower=0)
    df = df.drop(['YearBuilt', 'YearRemodAdd', 'GarageYrBlt'], axis=1)

    # F. Catégories ABSENTES = 'None'
    for c in ['Alley', 'Fence', 'MiscFeature', 'GarageType', 'MasVnrType']:
        if c in df.columns:
            df[c] = df[c].fillna('None')

    return df


# --- Chargement
train_df = pd.read_csv('Data/train.csv')
test_df = pd.read_csv('Data/test.csv')

# --- Suppression des 2 outliers (UNIQUEMENT sur le train)
train_df = train_df.drop(train_df[train_df['GrLivArea'] > 4000].index)

# --- Feature engineering
train_df = feature_engineering(train_df)
test_df = feature_engineering(test_df)

# --- Séparation X / y (cible en log pour s'aligner sur le RMSLE)
X_train = train_df.drop(['Id', 'SalePrice'], axis=1)
y_train = pd.Series(np.log1p(train_df['SalePrice']), index=train_df.index)

X_test = test_df.drop(['Id'], axis=1)
test_ids = test_df['Id']

print("X_train :", X_train.shape, "| X_test :", X_test.shape)

X_train : (1456, 88) | X_test : (1459, 88)


In [15]:
# 3. PREPROCESSOR
# Numérique : imputation médiane (PAS de scaling, inutile pour les arbres).
# Catégoriel : TOUTES les nominales en Target Encoding (réajusté par pli => pas de fuite).
#   Les catégories "absentes" ont déjà été remplies par 'None' à l'étape 2 ;
#   l'imputation 'most_frequent' ne concerne donc que les vrais manquants (MSZoning, SaleType...).

numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

print("Numériques :", len(numeric_features), "| Catégorielles (Target Encoding) :", len(categorical_features))


def build_preprocessor():
    """Reconstruit un preprocessor neuf (réajusté sur chaque pli => pas de fuite)."""
    num_t = SimpleImputer(strategy='median')

    cat_t = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('target_encoder', TargetEncoder(smoothing=10, handle_unknown='value', handle_missing='value'))
    ])

    return ColumnTransformer(transformers=[
        ('num', num_t, numeric_features),
        ('cat', cat_t, categorical_features),
    ])

Numériques : 59 | Catégorielles (Target Encoding) : 29


In [16]:
# 4. TUNING OPTUNA (CV manuelle + early stopping)
# Le preprocessor (dont le TargetEncoder) est réajusté sur chaque pli d'entraînement
# => aucune fuite de la cible vers le pli de validation.

optuna.logging.set_verbosity(optuna.logging.WARNING)
N_SPLITS = 5


def cross_validate(params, return_iters=False):
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    rmses, best_iters = [], []
    for tr_idx, val_idx in kf.split(X_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        pre = build_preprocessor()
        X_tr_t = pre.fit_transform(X_tr, y_tr)
        X_val_t = pre.transform(X_val)

        model = XGBRegressor(
            n_estimators=3000,            # plafond élevé ; l'early stopping choisit le bon nombre
            early_stopping_rounds=50,
            tree_method='hist',
            random_state=42, n_jobs=-1,
            **params
        )
        model.fit(X_tr_t, y_tr, eval_set=[(X_val_t, y_val)], verbose=False)
        rmses.append(np.sqrt(mean_squared_error(y_val, model.predict(X_val_t))))
        best_iters.append(model.best_iteration)

    if return_iters:
        return np.mean(rmses), np.mean(best_iters)
    return np.mean(rmses)


def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 2, 6),
        'min_child_weight': trial.suggest_int('min_child_weight', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 0.8),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 10.0, log=True),
    }
    return cross_validate(params)


print("Lancement d'Optuna...")
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(objective, n_trials=80, show_progress_bar=True)

print("\n--- RÉSULTATS ---")
print(f"Meilleur RMSE (CV, échelle log) : {study.best_value:.5f}")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

Lancement d'Optuna...


Best trial: 48. Best value: 0.112753: 100%|██████████| 80/80 [09:00<00:00,  6.75s/it]


--- RÉSULTATS ---
Meilleur RMSE (CV, échelle log) : 0.11275
  max_depth: 4
  min_child_weight: 5
  learning_rate: 0.022458876023846695
  subsample: 0.5023147510645035
  colsample_bytree: 0.4901429374583633
  reg_alpha: 0.11544066323643319
  reg_lambda: 0.0286912028778077


In [17]:
# 5. MODÈLE FINAL : n_estimators optimal puis réentraînement sur TOUT le train

best_params = study.best_params

_, avg_best_iter = cross_validate(best_params, return_iters=True)
final_n_estimators = int(avg_best_iter * 1.1)   # léger surplus : on entraîne sur plus de données
print(f"n_estimators retenu : {final_n_estimators}")

final_preprocessor = build_preprocessor()
X_train_t = final_preprocessor.fit_transform(X_train, y_train)

final_model = XGBRegressor(
    n_estimators=final_n_estimators,
    tree_method='hist',
    random_state=42, n_jobs=-1,
    **best_params
)
final_model.fit(X_train_t, y_train)
print("Modèle final entraîné.")

n_estimators retenu : 538
Modèle final entraîné.


In [18]:
# 6. SOUMISSION KAGGLE

X_test_t = final_preprocessor.transform(X_test)
final_predictions = np.expm1(final_model.predict(X_test_t))   # retour en dollars

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': final_predictions})
fichier = 'submission_M4_XGBoost_Optuna.csv'
submission.to_csv(fichier, index=False)
print(f"Fichier '{fichier}' prêt à soumettre.")
submission.head()

Fichier 'submission_M4_XGBoost_Optuna.csv' prêt à soumettre.


,Id,SalePrice
0,1461,121998.820312
1,1462,159912.218750
2,1463,181412.265625
3,1464,190673.000000
4,1465,190348.500000
